In [1]:
# NPS data cleaning pipeline: combines IRMA visitation, fuzzy-matched entrance fees, and PDF-extracted Visitor Spending Effects into analysis-ready CSVs.

# This notebook performs end-to-end ETL on three NPS data sources: 46 years of IRMA monthly visitation data (219K rows), current entrance fee data manually compiled 
# from NPS sources, and 8 years of Visitor Spending Effects reports extracted from PDFs.

# Park names are reconciled across systems using rapidfuzz with token_sort_ratio scoring plus manual overrides for redesignations. 
# Output: three cleaned CSVs ready for PostgreSQL load and downstream analysis.

# Outputs are saved to data/processed/.

In [2]:
import pandas as pd
import os
import re

from rapidfuzz import process, fuzz

import pdfplumber

In [3]:
#set paths
raw_path = os.path.join('..','data', 'raw') #change when converting back to .py file
processed_path = os.path.join('..','data', 'processed')

## Step 1: loading and inspecting & joining the 2 datasets for irma visitation data

In [4]:
df1 = pd.read_csv(os.path.join(raw_path, 'irma_visitation_raw_1.csv'))
df2 = pd.read_csv(os.path.join(raw_path, 'irma_visitation_raw_2.csv'))

/var/folders/lv/4yk6sb4x0cndf5swhmmpnhp40000gn/T/ipykernel_84876/2247076289.py:2: DtypeWarning: Columns (0: NonRecreationOvernightStays) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv(os.path.join(raw_path, 'irma_visitation_raw_2.csv'))


In [5]:
print('==Dataset 1 1979-2004==')
print(f"Year range: {df1['Year'].min()} to {df1['Year'].max()}") 
print(f"Shape: {df1.shape}")
print(f"Columns {df1.columns}")

==Dataset 1 1979-2004==
Year range: 1979 to 2010
Shape: (125172, 35)
Columns Index(['ParkName', 'UnitCode', 'ParkType', 'Region', 'State', 'Year', 'Month',
       'RecreationVisits', 'NonRecreationVisits', 'RecreationHours',
       'NonRecreationHours', 'ConcessionerLodging', 'ConcessionerCamping',
       'TentCampers', 'RVCampers', 'Backcountry',
       'NonRecreationOvernightStays', 'MiscellaneousOvernightStays',
       'ParkNameTotal', 'UnitCodeTotal', 'ParkTypeTotal', 'RegionTotal',
       'StateTotal', 'YearTotal', 'RecreationVisitsTotal',
       'NonRecreationVisitsTotal', 'RecreationHoursTotal',
       'NonRecreationHoursTotal', 'ConcessionerLodgingTotal',
       'ConcessionerCampingTotal', 'TentCampersTotal', 'RVCampersTotal',
       'BackcountryTotal', 'NonRecreationOvernightStaysTotal',
       'MiscellaneousOvernightStaysTotal'],
      dtype='str')


In [6]:
print(df1.dtypes)
df1.head()

ParkName                              str
UnitCode                              str
ParkType                              str
Region                                str
State                                 str
Year                                int64
Month                               int64
RecreationVisits                      str
NonRecreationVisits                   str
RecreationHours                       str
NonRecreationHours                    str
ConcessionerLodging                   str
ConcessionerCamping                   str
TentCampers                           str
RVCampers                             str
Backcountry                           str
NonRecreationOvernightStays           str
MiscellaneousOvernightStays           str
ParkNameTotal                         str
UnitCodeTotal                         str
ParkTypeTotal                         str
RegionTotal                           str
StateTotal                            str
YearTotal                         

,ParkName,UnitCode,ParkType,Region,State,Year,Month,RecreationVisits,NonRecreationVisits,RecreationHours,...,NonRecreationVisitsTotal,RecreationHoursTotal,NonRecreationHoursTotal,ConcessionerLodgingTotal,ConcessionerCampingTotal,TentCampersTotal,RVCampersTotal,BackcountryTotal,NonRecreationOvernightStaysTotal,MiscellaneousOvernightStaysTotal
0,Abraham Lincoln Birthplace NHP,ABLI,National Historical Park,Southeast,KY,1979,1,"1,586",0,"1,586",...,0,"271,231",0,0,0,0,0,0,0,0
1,Abraham Lincoln Birthplace NHP,ABLI,National Historical Park,Southeast,KY,1979,2,"3,036",0,"3,036",...,0,"271,231",0,0,0,0,0,0,0,0
2,Abraham Lincoln Birthplace NHP,ABLI,National Historical Park,Southeast,KY,1979,3,"9,853",0,"9,853",...,0,"271,231",0,0,0,0,0,0,0,0
3,Abraham Lincoln Birthplace NHP,ABLI,National Historical Park,Southeast,KY,1979,4,"33,087",0,"33,087",...,0,"271,231",0,0,0,0,0,0,0,0
4,Abraham Lincoln Birthplace NHP,ABLI,National Historical Park,Southeast,KY,1979,5,"27,838",0,"27,838",...,0,"271,231",0,0,0,0,0,0,0,0


In [7]:
print("Dataset 2: (2005-2025)")
print(f"Year range: {df2['Year'].min()} to {df2['Year'].max()}") 
print(f"Shape: {df2.shape}")
print(f"Columns {df2.columns}")

Dataset 2: (2005-2025)
Year range: 2005 to 2025
Shape: (94419, 35)
Columns Index(['ParkName', 'UnitCode', 'ParkType', 'Region', 'State', 'Year', 'Month',
       'RecreationVisits', 'NonRecreationVisits', 'RecreationHours',
       'NonRecreationHours', 'ConcessionerLodging', 'ConcessionerCamping',
       'TentCampers', 'RVCampers', 'Backcountry',
       'NonRecreationOvernightStays', 'MiscellaneousOvernightStays',
       'ParkNameTotal', 'UnitCodeTotal', 'ParkTypeTotal', 'RegionTotal',
       'StateTotal', 'YearTotal', 'RecreationVisitsTotal',
       'NonRecreationVisitsTotal', 'RecreationHoursTotal',
       'NonRecreationHoursTotal', 'ConcessionerLodgingTotal',
       'ConcessionerCampingTotal', 'TentCampersTotal', 'RVCampersTotal',
       'BackcountryTotal', 'NonRecreationOvernightStaysTotal',
       'MiscellaneousOvernightStaysTotal'],
      dtype='str')


In [8]:
print(df2.dtypes)
df2.head(3)

ParkName                               str
UnitCode                               str
ParkType                               str
Region                                 str
State                                  str
Year                                 int64
Month                                int64
RecreationVisits                       str
NonRecreationVisits                    str
RecreationHours                        str
NonRecreationHours                     str
ConcessionerLodging                    str
ConcessionerCamping                    str
TentCampers                            str
RVCampers                              str
Backcountry                            str
NonRecreationOvernightStays         object
MiscellaneousOvernightStays            str
ParkNameTotal                          str
UnitCodeTotal                          str
ParkTypeTotal                          str
RegionTotal                            str
StateTotal                             str
YearTotal  

,ParkName,UnitCode,ParkType,Region,State,Year,Month,RecreationVisits,NonRecreationVisits,RecreationHours,...,NonRecreationVisitsTotal,RecreationHoursTotal,NonRecreationHoursTotal,ConcessionerLodgingTotal,ConcessionerCampingTotal,TentCampersTotal,RVCampersTotal,BackcountryTotal,NonRecreationOvernightStaysTotal,MiscellaneousOvernightStaysTotal
0,Abraham Lincoln Birthplace NHP,ABLI,National Historical Park,Southeast,KY,2005,1,"4,160",0,"4,160",...,0,"190,809",0,0,0,0,0,0,0,0
1,Abraham Lincoln Birthplace NHP,ABLI,National Historical Park,Southeast,KY,2005,2,"6,422",0,"6,422",...,0,"190,809",0,0,0,0,0,0,0,0
2,Abraham Lincoln Birthplace NHP,ABLI,National Historical Park,Southeast,KY,2005,3,"10,737",0,"10,737",...,0,"190,809",0,0,0,0,0,0,0,0


In [9]:
visit_df = pd.concat([df1,df2], ignore_index=True)
print(f"Combined Shape: {visit_df.shape}")
print(f"Year range: {visit_df['Year'].min()} to {visit_df['Year'].max()}") 


Combined Shape: (219591, 35)
Year range: 1979 to 2025


In [10]:
visit_df['ParkType'].unique()

<StringArray>
[    'National Historical Park',                'National Park',
            'National Monument',       'National Historic Site',
     'National Recreation Area',         'National Battlefield',
           'National Lakeshore',            'National Memorial',
            'National Seashore',            'National Preserve',
               'National River',             'National Parkway',
 'National Wild & Scenic River',    'National Battlefield Site',
                 'Park (Other)',       'National Military Park',
             'National Reserve',    'National Battlefield Park',
        'National Scenic Trail',  'International Historic Site']
Length: 20, dtype: str

## Step 2: Check & Clean visitation data

In [11]:
# dropping from column 18 onwards as they are just aggregates, Total of prev columns
total_cols = [col for col in visit_df.columns if 'Total' in col]
print(f"Dropping {len(total_cols)} columns: {total_cols}")

visit_df = visit_df.drop(columns=total_cols)
print(f"Visitation dataset shape: {visit_df.shape}")
print(f"New Columns: {visit_df.columns}")

Dropping 17 columns: ['ParkNameTotal', 'UnitCodeTotal', 'ParkTypeTotal', 'RegionTotal', 'StateTotal', 'YearTotal', 'RecreationVisitsTotal', 'NonRecreationVisitsTotal', 'RecreationHoursTotal', 'NonRecreationHoursTotal', 'ConcessionerLodgingTotal', 'ConcessionerCampingTotal', 'TentCampersTotal', 'RVCampersTotal', 'BackcountryTotal', 'NonRecreationOvernightStaysTotal', 'MiscellaneousOvernightStaysTotal']
Visitation dataset shape: (219591, 18)
New Columns: Index(['ParkName', 'UnitCode', 'ParkType', 'Region', 'State', 'Year', 'Month',
       'RecreationVisits', 'NonRecreationVisits', 'RecreationHours',
       'NonRecreationHours', 'ConcessionerLodging', 'ConcessionerCamping',
       'TentCampers', 'RVCampers', 'Backcountry',
       'NonRecreationOvernightStays', 'MiscellaneousOvernightStays'],
      dtype='str')


In [12]:
numeric_cols = visit_df.columns[7:]
print(f"Converting these columns from string to numeric: \n{numeric_cols}")

for col in numeric_cols:
    visit_df[col] = visit_df[col].str.replace(',','').astype(float)

print(visit_df.dtypes)

Converting these columns from string to numeric: 
Index(['RecreationVisits', 'NonRecreationVisits', 'RecreationHours',
       'NonRecreationHours', 'ConcessionerLodging', 'ConcessionerCamping',
       'TentCampers', 'RVCampers', 'Backcountry',
       'NonRecreationOvernightStays', 'MiscellaneousOvernightStays'],
      dtype='str')
ParkName                           str
UnitCode                           str
ParkType                           str
Region                             str
State                              str
Year                             int64
Month                            int64
RecreationVisits               float64
NonRecreationVisits            float64
RecreationHours                float64
NonRecreationHours             float64
ConcessionerLodging            float64
ConcessionerCamping            float64
TentCampers                    float64
RVCampers                      float64
Backcountry                    float64
NonRecreationOvernightStays    float64
Misc

In [13]:
# df[df['column'] == 'value'] pattern is how you filter in pandas.

print(f'Total rows: {len(visit_df)}')
print(f'Unique Parks: {visit_df["ParkName"].nunique()} (IRMA says only 406/433 parks record visitation data)')
print(f'National Parks: {visit_df[visit_df["ParkType"] == "National Park"]["ParkName"].nunique()}')
visit_df.head()

Total rows: 219591
Unique Parks: 406 (IRMA says only 406/433 parks record visitation data)
National Parks: 63


,ParkName,UnitCode,ParkType,Region,State,Year,Month,RecreationVisits,NonRecreationVisits,RecreationHours,NonRecreationHours,ConcessionerLodging,ConcessionerCamping,TentCampers,RVCampers,Backcountry,NonRecreationOvernightStays,MiscellaneousOvernightStays
0,Abraham Lincoln Birthplace NHP,ABLI,National Historical Park,Southeast,KY,1979,1,1586.0,0.0,1586.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Abraham Lincoln Birthplace NHP,ABLI,National Historical Park,Southeast,KY,1979,2,3036.0,0.0,3036.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Abraham Lincoln Birthplace NHP,ABLI,National Historical Park,Southeast,KY,1979,3,9853.0,0.0,9853.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Abraham Lincoln Birthplace NHP,ABLI,National Historical Park,Southeast,KY,1979,4,33087.0,0.0,33087.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Abraham Lincoln Birthplace NHP,ABLI,National Historical Park,Southeast,KY,1979,5,27838.0,0.0,27838.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
# Checking for Missing values
print("====Checking for missing values====")
print(visit_df.isnull().sum())

====Checking for missing values====
ParkName                           0
UnitCode                           0
ParkType                           0
Region                             0
State                           2316
Year                               0
Month                              0
RecreationVisits                   0
NonRecreationVisits                0
RecreationHours                    0
NonRecreationHours                 0
ConcessionerLodging                0
ConcessionerCamping                0
TentCampers                        0
RVCampers                          0
Backcountry                        0
NonRecreationOvernightStays    12499
MiscellaneousOvernightStays        0
dtype: int64


In [15]:
#which parks have missing state values?
missing_states = visit_df[visit_df['State'].isnull()]['ParkName'].unique()
print(f"Parks with missing states: {len(missing_states)}")
print(missing_states)


Parks with missing states: 32
<StringArray>
[                             'Alagnak WR',
                              'Amache NHS',
                         'Appalachian NST',
        'Belmont-Paul Women's Equality NM',
               'Boston Harbor Islands NRA',
                          'Camp Nelson NM',
              'Carter G. Woodson Home NHS',
                      'Cesar E. Chavez NM',
       'Charles Young Buffalo Soldiers NM',
                'Dwight D. Eisenhower MEM',
    'Emmett Till and Mamie Till-Mobley NM',
                         'First State NHP',
                      'Harriet Tubman NHP',
 'Harriet Tubman Underground Railroad NHP',
                             'Ice Age NST',
            'Katahdin Woods and Waters NM',
                            'Keweenaw NHP',
                   'Manhattan Project NHP',
         'Medgar and Myrlie Evers Home NM',
             'Mill Springs Battlefield NM',
                            'Minidoka NHS',
                'Paterson Great 

In [16]:
#visit_df[visit_df['ParkName']]['State'].unique

ma = visit_df[visit_df['State'].isnull()][['ParkName','State','Year']]
print (f'Missing state data is only from year: {ma['Year'].max()} - {ma['Year'].min()}')
print(f'Total Parks classified by NPS before 2013: {visit_df[visit_df['Year']<2013]['ParkName'].nunique()}')
print(f'Total Parks classified by NPS in and after 2013: {visit_df[visit_df['Year']>=2013]['ParkName'].nunique()}')
print(f'Total Parks from 1973: {visit_df['ParkName'].nunique()} (some parks are reported only for few years or combined with others)')
print('\n Parks with no states listed')
visit_df[visit_df['State'].isnull()][['ParkName','State']].drop_duplicates()

Missing state data is only from year: 2025 - 2013
Total Parks classified by NPS before 2013: 373
Total Parks classified by NPS in and after 2013: 400
Total Parks from 1973: 406 (some parks are reported only for few years or combined with others)

 Parks with no states listed


,ParkName,State
126372,Alagnak WR,NaN
126936,Amache NHS,NaN
128472,Appalachian NST,NaN
130500,Belmont-Paul Women's Equality NM,NaN
134148,Boston Harbor Islands NRA,NaN
135732,Camp Nelson NM,NaN
138816,Carter G. Woodson Home NHS,NaN
140184,Cesar E. Chavez NM,NaN
141347,Charles Young Buffalo Soldiers NM,NaN
148535,Dwight D. Eisenhower MEM,NaN


In [17]:
pre_2013 = set(visit_df[visit_df['Year'] < 2013]['ParkName'].unique())
post_2013 = set(visit_df[visit_df['Year'] >= 2013]['ParkName'].unique())
print('Parks only in pre-2013 (stopped reporting):')
print(pre_2013 - post_2013)

Parks only in pre-2013 (stopped reporting):
{'Brices Cross Roads', 'National Capital Parks Combined', 'John F. Kennedy Center For Pa', 'National Visitor Center', 'Tupelo NBS', 'Oklahoma City NMEM'}


In [18]:
state_fixes = {
    'Alagnak WR': 'AK',
    'Amache NHS': 'CO',
    'Appalachian NST': 'WV',
    'Belmont-Paul Women\'s Equality NM': 'DC',
    'Boston Harbor Islands NRA': 'MA',
    'Camp Nelson NM': 'KY',
    'Carter G. Woodson Home NHS': 'DC',
    'Cesar E. Chavez NM': 'CA',
    'Charles Young Buffalo Soldiers NM': 'OH',
    'Dwight D. Eisenhower MEM': 'DC',
    'Emmett Till and Mamie Till-Mobley NM': 'MS',
    'First State NHP': 'DE',
    'Harriet Tubman NHP': 'NY',
    'Harriet Tubman Underground Railroad NHP': 'MD',
    'Ice Age NST': 'WI',
    'Katahdin Woods and Waters NM': 'ME',
    'Keweenaw NHP': 'MI',
    'Manhattan Project NHP': 'NM',
    'Medgar and Myrlie Evers Home NM': 'MS',
    'Mill Springs Battlefield NM': 'KY',
    'Minidoka NHS': 'ID',
    'Paterson Great Falls NHP': 'NJ',
    'Pullman NHP': 'IL',
    'Reconstruction Era NHP': 'SC',
    'Rosie The Riveter WWII Home Front NHP': 'CA',
    'Ste. Genevieve NHP': 'MO',
    'Stonewall NM': 'NY',
    'Tule Lake NM': 'CA',
    'Tule Springs Fossil Beds NM': 'NV',
    'Valles Caldera NPRES': 'NM',
    'Waco Mammoth NM': 'TX',
    'World War I MEM': 'DC'
}

# apply the fixes
for park, state in state_fixes.items():
    visit_df.loc[visit_df['ParkName'] == park, 'State'] = state

# verify no more nulls
print(f'Missing states remaining: {visit_df['State'].isnull().sum()}')

Missing states remaining: 0


In [19]:
# Filling up the remaining 2 columns with 0 instead of nulls since its just data that wasn't recorded for those parks

visit_df['Backcountry'] = visit_df['Backcountry'].fillna(0)
visit_df['NonRecreationOvernightStays'] = visit_df['NonRecreationOvernightStays'].fillna(0)

# verify no nulls remain anywhere
print(f'Remaining nulls:\n{visit_df.isnull().sum()}')

Remaining nulls:
ParkName                       0
UnitCode                       0
ParkType                       0
Region                         0
State                          0
Year                           0
Month                          0
RecreationVisits               0
NonRecreationVisits            0
RecreationHours                0
NonRecreationHours             0
ConcessionerLodging            0
ConcessionerCamping            0
TentCampers                    0
RVCampers                      0
Backcountry                    0
NonRecreationOvernightStays    0
MiscellaneousOvernightStays    0
dtype: int64


In [20]:
print('Saving the cleaned visitation data to processed folder')

visit_df.to_csv(os.path.join(processed_path, 'visitation_cleaned.csv'), index = False)

Saving the cleaned visitation data to processed folder


In [21]:
visit_df = pd.read_csv(os.path.join(processed_path, 'visitation_cleaned.csv'))
print(f'Before dedup: {visit_df.shape}')

# drop exact duplicates
visit_df = visit_df.drop_duplicates(subset=['ParkName','UnitCode','Year','Month'], keep='first')
print(f'After dedup: {visit_df.shape}')

# resave
visit_df.to_csv(os.path.join(processed_path, 'visitation_cleaned.csv'), index=False)

Before dedup: (219591, 18)
After dedup: (193696, 18)


In [22]:
# strip whitespaces from all columns
visit_df = pd.read_csv(os.path.join(processed_path, 'visitation_cleaned.csv'))

str_cols = visit_df.select_dtypes(include='object').columns
for col in str_cols:
    visit_df[col] = visit_df[col].str.strip()

# Verify
print(visit_df['Region'].unique())

# resave
visit_df.to_csv(os.path.join(processed_path, 'visitation_cleaned.csv'), index=False)

/var/folders/lv/4yk6sb4x0cndf5swhmmpnhp40000gn/T/ipykernel_84876/1130062714.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = visit_df.select_dtypes(include='object').columns


<StringArray>
[       'Southeast',        'Northeast',          'Midwest',
    'Intermountain',           'Alaska', 'National Capital',
     'Pacific West']
Length: 7, dtype: str


## Step 3: Cleaning entrance fee data

In [23]:
fees_df = pd.read_csv(os.path.join(raw_path,'park_entrance_fees.csv'))
print(f'Fees dataset shape: {fees_df.shape}')
print(fees_df.dtypes)
fees_df.head(15)

Fees dataset shape: (118, 4)
Name           str
Location       str
Typical fee    str
Fee type       str
dtype: object


,Name,Location,Typical fee,Fee type
0,Acadia National Park,Maine,$20,per-person
1,Adams National Historical Park,Massachusetts,$15,per-person
2,Antietam National Battlefield,Maryland,$10,per-person
3,Arches National Park,Utah,$15,per-person
4,Assateague Island National Seashore,Maryland,$25,per-vehicle
5,Assateague Island National Seashore,Virginia,$25,per-vehicle
6,Badlands National Park,South Dakota,$15,per-person
7,Bandelier National Monument,New Mexico,$15,per-person
8,Big Bend National Park,Texas,$15,per-person
9,Black Canyon of the Gunnison National Park,Colorado,$15,per-person


In [24]:
# renaming columns
fees_df = fees_df.rename(columns={
    'Name': 'ParkName',
    'Location': 'State',
    'Typical fee': 'entrance_fee',
    'Fee type': 'fee_type'
})

# ⬇️ can only run this once since column is converted to float and then str commands don't apply to it if run once
#converting fees column from string to float after removing whitespaces and ＄sign
fees_df['entrance_fee'] = fees_df['entrance_fee'].str.strip().str.replace('$','').astype(float) 

#drop duplicates of parks (multi-state entries), keep first
fees_df = fees_df.drop_duplicates(subset='ParkName', keep='first')
print(f'after dropping duplicates {fees_df.shape}')

after dropping duplicates (105, 4)


In [25]:
#creating new column converting per-vehicle to per-person fees by using NPS formula
print('Using NPS formula of per-vehicle / 2.5 = per-person, we create new column')
fees_df['per_person_fee'] = fees_df.apply(
    lambda row: row['entrance_fee'] / 2.5 if row['fee_type'] == 'per-vehicle' else row['entrance_fee'],
    axis = 1
)

fees_df.head(10)


Using NPS formula of per-vehicle / 2.5 = per-person, we create new column


,ParkName,State,entrance_fee,fee_type,per_person_fee
0,Acadia National Park,Maine,20.0,per-person,20.0
1,Adams National Historical Park,Massachusetts,15.0,per-person,15.0
2,Antietam National Battlefield,Maryland,10.0,per-person,10.0
3,Arches National Park,Utah,15.0,per-person,15.0
4,Assateague Island National Seashore,Maryland,25.0,per-vehicle,10.0
6,Badlands National Park,South Dakota,15.0,per-person,15.0
7,Bandelier National Monument,New Mexico,15.0,per-person,15.0
8,Big Bend National Park,Texas,15.0,per-person,15.0
9,Black Canyon of the Gunnison National Park,Colorado,15.0,per-person,15.0
10,Bryce Canyon National Park,Utah,20.0,per-person,20.0


In [26]:
#fuzzy matching park names to rempa them to IRMA standard

#get unique parks from both datasets
fee_parks = fees_df['ParkName'].unique()
irma_parks = visit_df['ParkName'].unique()

#for each fee park, find the best match in IRMA data
matches = []
for park in fee_parks:
    best_match, score, _ = process.extractOne(park, irma_parks, scorer=fuzz.token_sort_ratio) #process.extractOne() returns three values: the best match, the score, and the index position of that match in the list. We need the match and score but don't care about the index. The _ is a Python convention that says "I know there's a value here but I'm intentionally ignoring it." It's just a throwaway variable.
    matches.append({
        'fee_name': park,
        'irma_match': best_match,
        'score': score
    })

match_df = pd.DataFrame(matches).sort_values('score')

#reviewing low confidence matches (needs manual review)
print('=== Matches below 70 Score (likely wrong)===')
print(match_df[match_df['score'] < 70].to_string(index=False))
print('=== Matches 70-85 Score (verify these)===')
print(match_df[(match_df['score'] >= 70) & (match_df['score'] < 85)].to_string(index=False))
print('=== Matches above 85 Score===')
print(match_df[match_df['score'] >= 85].to_string(index=False))


=== Matches below 70 Score (likely wrong)===
                                          fee_name                        irma_match     score
Fort McHenry National Monument and Historic Shrine Fort Sumter and Fort Moultrie NHP 50.602410
                Lake Mead National Recreation Area                     Lake Mead NRA 51.063830
                    Rio Grande Wild & Scenic River                   Rio Grande W&SR 53.333333
              Glen Canyon National Recreation Area                   Glen Canyon NRA 54.901961
              Whiskeytown National Recreation Area                   Whiskeytown NRA 54.901961
                 Fort Davis National Historic Site                    Fort Davis NHS 55.319149
                       Bandelier National Monument               Washington Monument 56.521739
                       Lava Beds National Monument               Washington Monument 56.521739
                        Dinosaur National Monument               Washington Monument 57.777778
     

In [27]:
# need to convert suffixes to abbreviations to make the fuzzy match more accurate

#order matters here - longer suffixes first so they match before the shorter ones

suffix_map = {
    'National Park and Preserve': 'NP & PRES',
    'National Historical Park': 'NHP',
    'National Historic Site': 'NHS',
    'National Military Park': 'NMP',
    'National Battlefield Park': 'NBP',
    'National Recreation Area': 'NRA',
    'National Monument': 'NM',
    'National Battlefield': 'NB',
    'National Seashore': 'NS',
    'National Lakeshore': 'NL',
    'National Preserve': 'NPRES',
    'National Memorial': 'NMEM',
    'National Park': 'NP',
    'Wild & Scenic River': 'W&SR'
}

#apply suffixes to the fees_df

fees_df['irma_name'] = fees_df['ParkName']

for full_suffix, abbrev in suffix_map.items():
    fees_df['irma_name'] = fees_df['irma_name'].str.replace(full_suffix, abbrev, regex=False) #regex=False means treat the search string as literal text, not a pattern -> , *, +, (, ) have special meanings for pattern matching and DON'T take that into account here

fees_df[['ParkName','irma_name']].head(20)


,ParkName,irma_name
0,Acadia National Park,Acadia NP
1,Adams National Historical Park,Adams NHP
2,Antietam National Battlefield,Antietam NB
3,Arches National Park,Arches NP
4,Assateague Island National Seashore,Assateague Island NS
6,Badlands National Park,Badlands NP
7,Bandelier National Monument,Bandelier NM
8,Big Bend National Park,Big Bend NP
9,Black Canyon of the Gunnison National Park,Black Canyon of the Gunnison NP
10,Bryce Canyon National Park,Bryce Canyon NP


In [28]:
#now retrying with abbreviated suffixes

#fuzzy matching park names to rempa them to IRMA standard

#get unique parks from both datasets
fee_parks = fees_df['irma_name'].unique()
irma_parks = visit_df['ParkName'].unique()

#for each fee park, find the best match in IRMA data
matches = []
for park in fee_parks:
    best_match, score, _ = process.extractOne(park, irma_parks, scorer=fuzz.token_sort_ratio) 
    matches.append({
        'fee_name': park,
        'irma_match': best_match,
        'score': score
    })

match_df = pd.DataFrame(matches).sort_values('score')

#reviewing low confidence matches (needs manual review)
print('=== Matches below 70 Score (likely wrong)===')
print(match_df[match_df['score'] < 70].to_string(index=False))
print('\n=== Matches 70-85 Score (verify these)===')
print(match_df[(match_df['score'] >= 70) & (match_df['score'] < 85)].to_string(index=False))
print('\n=== Matches above 85 Score===')
print(match_df[match_df['score'] >= 85].to_string(index=False))


=== Matches below 70 Score (likely wrong)===
                                        fee_name                        irma_match     score
             Fort McHenry NM and Historic Shrine              Fort McHenry NM & HS 61.818182
                                 C & O Canal NHP       Chesapeake & Ohio Canal NHP 61.904762
Perry's Victory and International Peace Memorial Perry's Victory & Intl. Peace MEM 69.135802

=== Matches 70-85 Score (verify these)===
           fee_name               irma_match     score
   Great Falls Park Paterson Great Falls NHP 75.000000
Lewis and Clark NHP        Lewis & Clark NHP 83.333333

=== Matches above 85 Score===
                         fee_name                        irma_match      score
           Craters of the Moon NM     Craters of the Moon NM & PRES  86.274510
  Chickamauga and Chattanooga NMP     Chickamauga & Chattanooga NMP  90.000000
         Pu'uhonua O Honaunau NHP          Pu'uhonua o Honaunau NHP  91.666667
    Little Bighorn Battlefie

In [29]:
#manually fixing the 5 non 100 score ones

manual_fixes = {
    'Fort McHenry NM and Historic Shrine': 'Fort McHenry NM & HS',
    'C & O Canal NHP': 'Chesapeake & Ohio Canal NHP',
    "Perry's Victory and International Peace Memorial": "Perry's Victory & Intl. Peace MEM",
    'Great Falls Park': 'Great Falls Park',
    'Lewis and Clark NHP': 'Lewis & Clark NHP',
    'Craters of the Moon NM': 'Craters of the Moon NM & PRES',
    'Chickamauga and Chattanooga NMP': 'Chickamauga & Chattanooga NMP',
    "Pu'uhonua O Honaunau NHP": "Pu'uhonua o Honaunau NHP"
}

for old_name, new_name in manual_fixes.items():
    fees_df.loc[fees_df['irma_name'] == old_name, 'irma_name'] = new_name

#replacing all the park names in fees_df with the updated matched IRMA names of irma_parks
fees_df['ParkName'] = fees_df['irma_name']
fees_df = fees_df.drop(columns=['irma_name'])

#verifying all fee parks exist in IRMA dataset
temp_fee_parks = set(fees_df['ParkName'].unique())
temp_irma_parks = set(visit_df['ParkName'].unique())

unmatched = temp_fee_parks - temp_irma_parks
print(f'Number of unmatched parks {len(unmatched)}')
if unmatched:
    print(unmatched)





Number of unmatched parks 1
{'Great Falls Park'}


### Great Falls Park isn't listed under IRMA data i.e NPS doesn't track visitors so we'll remove it


In [30]:
#drop great falls
fees_df = fees_df[fees_df['ParkName'] != 'Great Falls Park']

# drop the Wikipedia state column entirely and pull from IRMA instead
# get unique park-state pairs from visit_df
irma_states = visit_df[['ParkName','State']].drop_duplicates()

#drop the old State column in fees_df and merge with IRMA states
fees_df = fees_df.drop(columns=['State'])
fees_df = fees_df.merge(irma_states, on='ParkName', how='left') #merge is SQL join in pandas, on is join key and how='left' means keep all rows from fees_df

#verify
print(f'Shape: {fees_df.shape}')
print(f'Null states: {fees_df['State'].isnull().sum()}')
fees_df

Shape: (104, 5)
Null states: 0


,ParkName,entrance_fee,fee_type,per_person_fee,State
0,Acadia NP,20.0,per-person,20.0,ME
1,Adams NHP,15.0,per-person,15.0,MA
2,Antietam NB,10.0,per-person,10.0,MD
3,Arches NP,15.0,per-person,15.0,UT
4,Assateague Island NS,25.0,per-vehicle,10.0,MD
...,...,...,...,...,...
99,Wright Brothers NMEM,10.0,per-person,10.0,NC
100,Wupatki NM,15.0,per-person,15.0,AZ
101,Yellowstone NP,20.0,per-person,20.0,WY
102,Yosemite NP,20.0,per-person,20.0,CA


In [31]:
#rearraning state to 2nd column
temp_col = fees_df.pop('State')
fees_df.insert(1,'State', temp_col)
fees_df

,ParkName,State,entrance_fee,fee_type,per_person_fee
0,Acadia NP,ME,20.0,per-person,20.0
1,Adams NHP,MA,15.0,per-person,15.0
2,Antietam NB,MD,10.0,per-person,10.0
3,Arches NP,UT,15.0,per-person,15.0
4,Assateague Island NS,MD,25.0,per-vehicle,10.0
...,...,...,...,...,...
99,Wright Brothers NMEM,NC,10.0,per-person,10.0
100,Wupatki NM,AZ,15.0,per-person,15.0
101,Yellowstone NP,WY,20.0,per-person,20.0
102,Yosemite NP,CA,20.0,per-person,20.0


In [32]:
fees_df.to_csv(os.path.join(processed_path,'fees_cleaned.csv'), index=False)
print(f'Saved cleaned fees data: {fees_df.shape}')
fees_df.head(10)

Saved cleaned fees data: (104, 5)


,ParkName,State,entrance_fee,fee_type,per_person_fee
0,Acadia NP,ME,20.0,per-person,20.0
1,Adams NHP,MA,15.0,per-person,15.0
2,Antietam NB,MD,10.0,per-person,10.0
3,Arches NP,UT,15.0,per-person,15.0
4,Assateague Island NS,MD,25.0,per-vehicle,10.0
5,Badlands NP,SD,15.0,per-person,15.0
6,Bandelier NM,NM,15.0,per-person,15.0
7,Big Bend NP,TX,15.0,per-person,15.0
8,Black Canyon of the Gunnison NP,CO,15.0,per-person,15.0
9,Bryce Canyon NP,UT,20.0,per-person,20.0


## Step 4: Getting Visitor Spending Data (2024)

In [33]:
# load the VSE PDF
pdf_path = os.path.join(raw_path, 'NPS_2024_Visitor_Spending_Effects.pdf')
pdf = pdfplumber.open(pdf_path)

#see how many pages and what's on them
print(f'Total pages: {len(pdf.pages)}')

#look at page 1 to understand the structure
print('\nFirst 500 characters of page 1')
print(pdf.pages[0].extract_text()[:500])             # extract_text() gets all the text for a page from page index specified. if you want only certain charcters use [5:200] range or whatevr

# find where the park level data table starts
for i in range(24,30):
    if i < len(pdf.pages):
        text = pdf.pages[i].extract_text()         #extract_table index gets specified rows, extract_table()[:4] gets first 4 rows form table
        if text:
            print(f'\n==Page{i+1} first 200 characters===')
            print(text[:200])


Total pages: 68

First 500 characters of page 1
National Park Service
U.S. Department of the Interior
Science Report NPS/SR—2025/353
https://doi.org/10.36967/2315417
2024 National Park Visitor Spending Effects
Economic Contributions to Local Communities, States,
and the Nation
Visitors watch an eruption on December 23, 2024, from an overlook northwest of Keanakākoʻi Crater at Hawaiʻi
Volcanoes National Park.
NPS / J. WEI

==Page25 first 200 characters===
worker for these sectors increased sharply between 2019–2023, driven by rising prices and lower
staffing levels. Therefore, the 2024 VSE total jobs estimate reflects an update to the underlying
econom

==Page26 first 200 characters===
Literature Cited
Cullinane Thomas, C., E. Cornachione, L. Koontz, and C. Keyes. 2019. National Park Service
Socioeconomic Pilot Survey – Visitor Spending Analysis. Natural Resource Report
NPS/NRSS/EQD

==Page27 first 200 characters===
Appendix A: Park-Level Visits, Spending, and Contributions to Local Eco

In [34]:
# park level data is from page 27-47 (index 26-46)
all_rows = []
for i in range(26,46):
    table = pdf.pages[i].extract_table()
    if table:
        for row in table:
            all_rows.append(row)            #extracting a list of lists where each list is a row in the table (also extracts the coloumn names as rows)

print(f'Total rows extracted: {len(all_rows)}')
print(f'First row (Header row): {all_rows[0]}')
print(f'2nd & 3rd data rows example: {all_rows[1:3]}')
print(f'Last row: {all_rows[-1:]}')


Total rows extracted: 422
First row (Header row): ['Park Unit', 'Recreation\nVisits', 'Visitor\nSpending\n($000s, $2024)', 'Jobs\nSupported', 'Labor Income\n($000s, $2024)', 'Value Added\n($000s, $2024)', 'Economic\nOutput\n($000s, $2024)']
2nd & 3rd data rows example: [['Abraham Lincoln Birthplace NHP', '249,166', '$17,534', '195', '$7,968', '$14,052', '$24,349'], ['Acadia NP A', '3,961,661', '$538,906', '5,304', '$243,133', '$459,921', '$744,545']]
Last row: [['Zion NP A', '4,946,592', '$774,194', '8,564', '$282,472', '$556,282', '$1,048,947']]


In [35]:
#converting to dataframe, first row is header, rest is data
vse_2024_df = pd.DataFrame(all_rows[1:], columns=all_rows[0])
print(f'Number of extra header rows removed: {vse_2024_df[vse_2024_df['Park Unit'] == 'Park Unit']['Park Unit'].count()}')
vse_2024_df = vse_2024_df[vse_2024_df['Park Unit'] != 'Park Unit'] # must remove repeated instances of header

# 2. Remove trailing footnote markers (A, B, A,B, A,C, A,B,C capital and small letter) in park names
vse_2024_df['Park Unit'] = vse_2024_df['Park Unit'].str.replace(r'\s+[A-Da-d](?:,[A-D])*$', '', regex=True)             #\s+ is one or more spaces $ is end of string

# 3. Remove state names in parentheses: "Manhattan Project (New Mexico) NHP" -> "Manhattan Project NHP"
vse_2024_df['Park Unit'] = vse_2024_df['Park Unit'].str.replace(r'\s*\(.*?\)\s*', ' ', regex=True).str.strip()

# 4. Fix "&" spacing: "NP&PRES" -> "NP & PRES", "NM&PRES" -> "NM & PRES"
vse_2024_df['Park Unit'] = vse_2024_df['Park Unit'].str.replace(r'&(?!\s)', '& ', regex=True)
vse_2024_df['Park Unit'] = vse_2024_df['Park Unit'].str.replace(r'(?<!\s)&', ' &', regex=True)

#renaming columns to be cleaner
vse_2024_df = vse_2024_df.rename(columns={
    'Park Unit': 'ParkName',
    'Recreation\nVisits': 'RecreationVisits',
    'Visitor\nSpending\n($000s, $2024)': 'VisitorSpending',
    'Jobs\nSupported': 'JobsSupported',
    'Labor Income\n($000s, $2024)': 'LaborIncome',
    'Value Added\n($000s, $2024)': 'ValueAdded',
    'Economic\nOutput\n($000s, $2024)': 'EconomicOutput'
})

vse_2024_df.insert(1, 'Year',2024)
print(f'Any null values:\n{vse_2024_df.isnull().sum()}')
print(f'New shape: {vse_2024_df.shape}')
print(vse_2024_df.dtypes)

vse_2024_df.head(10)

Number of extra header rows removed: 19
Any null values:
ParkName            0
Year                0
RecreationVisits    0
VisitorSpending     0
JobsSupported       0
LaborIncome         0
ValueAdded          0
EconomicOutput      0
dtype: int64
New shape: (402, 8)
ParkName              str
Year                int64
RecreationVisits      str
VisitorSpending       str
JobsSupported         str
LaborIncome           str
ValueAdded            str
EconomicOutput        str
dtype: object


,ParkName,Year,RecreationVisits,VisitorSpending,JobsSupported,LaborIncome,ValueAdded,EconomicOutput
0,Abraham Lincoln Birthplace NHP,2024,"249,166","$17,534",195,"$7,968","$14,052","$24,349"
1,Acadia NP,2024,"3,961,661","$538,906","5,304","$243,133","$459,921","$744,545"
2,Adams NHP,2024,"45,656","$3,213",31,"$1,804","$3,028","$4,691"
3,African Burial Ground NM,2024,"36,007","$2,534",21,"$1,410","$2,447","$3,606"
4,Agate Fossil Beds NM,2024,"19,395","$1,694",18,$484,$960,"$1,813"
5,Alagnak WR,2024,278,$14,0,$5,$9,$15
6,Alibates Flint Quarries NM,2024,"8,558",$602,6,$238,$428,$762
7,Allegheny Portage Railroad NHS,2024,"193,903","$13,645",148,"$6,644","$11,289","$18,998"
8,Amache NHS,2024,"4,771",$336,4,$111,$210,$379
9,Amistad NRA,2024,"832,294","$43,943",426,"$12,793","$24,920","$45,868"


In [36]:
vse_2024_df[60:75]

,ParkName,Year,RecreationVisits,VisitorSpending,JobsSupported,LaborIncome,ValueAdded,EconomicOutput
63,Castillo De San Marcos NM,2024,"579,825","$40,801",433,"$17,585","$32,726","$55,523"
64,Castle Clinton NM,2024,"3,822,759","$113,498",869,"$52,628","$91,132","$137,851"
65,Catoctin Mountain P,2024,"380,681","$11,749",91,"$4,744","$8,389","$13,178"
66,Cedar Breaks NM,2024,"722,834","$55,859",547,"$20,779","$38,877","$67,180"
67,Cesar E. Chavez NM,2024,"26,641","$1,875",18,$968,"$1,657","$2,618"
68,Chaco Culture NHP,2024,"37,840","$2,456",24,$940,"$1,724","$3,001"
69,Chamizal NMEM,2024,"6,627",$466,5,$174,$322,$585
70,Channel Islands NP,2024,"262,581","$17,742",161,"$9,228","$15,833","$24,863"
71,Charles Pinckney NHS,2024,"30,903","$2,175",22,$886,"$1,708","$2,830"
72,Charles Young Buffalo Soldiers NM,2024,"4,716",$332,4,$143,$249,$428


In [37]:
for col in vse_2024_df.columns:
    if vse_2024_df[col].dtype == 'str' and col not in ['ParkName', 'RecreationVisits', 'JobsSupported']:            #to handle multiple AND condtions 
        vse_2024_df[col] = vse_2024_df[col].str.replace(',','').str.replace('$','').astype(float) * 1000

for col in vse_2024_df.columns:
    if vse_2024_df[col].dtype == 'str' and col in ['RecreationVisits', 'JobsSupported']:            
        vse_2024_df[col] = vse_2024_df[col].str.replace(',','').str.replace('$','').astype(float)
vse_2024_df.head()

,ParkName,Year,RecreationVisits,VisitorSpending,JobsSupported,LaborIncome,ValueAdded,EconomicOutput
0,Abraham Lincoln Birthplace NHP,2024,249166.0,17534000.0,195.0,7968000.0,14052000.0,24349000.0
1,Acadia NP,2024,3961661.0,538906000.0,5304.0,243133000.0,459921000.0,744545000.0
2,Adams NHP,2024,45656.0,3213000.0,31.0,1804000.0,3028000.0,4691000.0
3,African Burial Ground NM,2024,36007.0,2534000.0,21.0,1410000.0,2447000.0,3606000.0
4,Agate Fossil Beds NM,2024,19395.0,1694000.0,18.0,484000.0,960000.0,1813000.0


In [38]:
#finally we check the park names if they match IRMA convention

temp_vse_parks = set(vse_2024_df['ParkName'].unique())
temp_irma_parks = set(visit_df['ParkName'].unique())

matched = temp_vse_parks & temp_irma_parks
unmatched = temp_vse_parks - temp_irma_parks

print(f'VSE parks: {len(temp_vse_parks)}')
print(f'Matches with IRMA: {len(matched)}\n{matched}')
print(f'Differences: {len(unmatched)}\n{unmatched}')

VSE parks: 398
Matches with IRMA: 339
{'Fort Matanzas NM', 'Springfield Armory NHS', 'Minidoka NHS', 'Hagerman Fossil Beds NM', 'Fort Caroline NMEM', 'Ice Age NST', 'Aztec Ruins NM', 'Cowpens NB', 'Gateway Arch NP', 'Abraham Lincoln Birthplace NHP', 'Pinnacles NP', 'Frederick Douglass NHS', 'Golden Gate NRA', 'Allegheny Portage Railroad NHS', 'Sagamore Hill NHS', 'Frederick Law Olmsted NHS', 'White Sands NP', 'Fort Larned NHS', 'Fort Sumter and Fort Moultrie NHP', 'Friendship Hill NHS', 'Saugus Iron Works NHS', 'Fort Laramie NHS', 'Badlands NP', 'Sleeping Bear Dunes NL', 'Hamilton Grange NMEM', 'Charles Pinckney NHS', 'Pictured Rocks NL', 'Martin Luther King, Jr. NHP', 'Effigy Mounds NM', 'Minute Man NHP', 'Petrified Forest NP', 'De Soto NMEM', 'Boston Harbor Islands NRA', 'Wupatki NM', 'Virgin Islands NP', 'Mammoth Cave NP', 'Niobrara NSR', 'Andersonville NHS', 'Cesar E. Chavez NM', 'Lincoln Home NHS', 'Fort Stanwix NM', 'Pea Ridge NMP', 'Herbert Hoover NHS', 'Big Cypress NPRES', 'San

In [39]:
#fuzzy matching the remaing - some are unicode character diff, comma ' differences so this will take care of that

matches = []
for temp_park in temp_vse_parks:
    best_match, score, _ = process.extractOne(temp_park, temp_irma_parks, scorer=fuzz.token_sort_ratio)
    matches.append({
        'vse name': temp_park,
        'IRMA match': best_match,
        'Score': score
    })

match_df = pd.DataFrame(matches).sort_values('Score')

print(f'===Scores below 100===\n{match_df[match_df['Score']<100].to_string(index=False)}')
print(f'\n===Perfect matches===\n{match_df[match_df['Score']==100].to_string(index=False)}')

===Scores below 100===
                                                 vse name                                 IRMA match     Score
                       Marsh - Billings - Rockefeller NHP             Marsh-Billings-Rockefeller NHP 62.500000
                                     Kaloko-Honokohau NHP                       Kaloko Honokohau NHP 65.000000
                             Yukon - Charley Rivers NPRES                 Yukon-Charley Rivers NPRES 66.666667
                                   War In The Pacific NHP                     War in the Pacific NHP 68.181818
Lyndon Baines Johnson Memorial Grove on the Potomac\nNMEM          LBJ Memorial Grove on the Potomac 69.662921
                                              Obed W & SR                                  Obed W&SR 70.000000
         Arlington House, The Robert E. Lee Memorial NMEM           Arlington House The R.E. Lee MEM 72.500000
                                    World War II Memorial                           World

In [40]:
#everything is correctly matched, just replacing it now with dictionaries

name_mapping = dict(zip(match_df['vse name'], match_df['IRMA match']))      #using a dictionary to map the vse names to the IRMA match using the dataframe we just created above

vse_2024_df['ParkName'] = vse_2024_df['ParkName'].replace(name_mapping)     #use replace() not map() as if no match found map() returns NaN but replace just leaves it alone -> no harm in rerunnign the cell

#rechecking the sets to see differences
temp_vse_parks = set(vse_2024_df['ParkName'])
temp_irma_parks = set(visit_df['ParkName'].unique())

matched = temp_vse_parks & temp_irma_parks
unmatched = temp_vse_parks - temp_irma_parks

print(f'VSE parks: {len(temp_vse_parks)}')
print(f'Matches with IRMA: {len(matched)}')
print(f'Differences: {len(unmatched)}')
print(unmatched)

vse_2024_df.isnull().sum()


VSE parks: 398
Matches with IRMA: 398
Differences: 0
set()


ParkName            0
Year                0
RecreationVisits    0
VisitorSpending     0
JobsSupported       0
LaborIncome         0
ValueAdded          0
EconomicOutput      0
dtype: int64

In [41]:
# there exists 3 duplicated parks, so we'll consolidate 
print(vse_2024_df.shape)
print('\nDuplicate parks:')
vse_2024_df[vse_2024_df['ParkName'].duplicated()]

(402, 8)

Duplicate parks:


,ParkName,Year,RecreationVisits,VisitorSpending,JobsSupported,LaborIncome,ValueAdded,EconomicOutput
111,Emmett Till and Mamie Till-Mobley NM,2024,1759.0,124000.0,1.0,44000.0,78000.0,142000.0
249,Manhattan Project NHP,2024,29511.0,2765000.0,27.0,1322000.0,2365000.0,3821000.0
250,Manhattan Project NHP,2024,19644.0,773000.0,6.0,259000.0,544000.0,847000.0
262,Minidoka NHS,2024,12495.0,879000.0,7.0,285000.0,639000.0,993000.0


In [42]:
vse_2024_df = vse_2024_df.groupby(['ParkName','Year'], as_index=False).sum()
print(f'After combining duplicates: {vse_2024_df.shape}')
print(f'Unique parks: {vse_2024_df['ParkName'].nunique()}')

After combining duplicates: (398, 8)
Unique parks: 398


In [43]:
vse_2024_df.to_csv(os.path.join(processed_path, 'VSE_2024_processed.csv'),index=False)
vse_2024_df.head()

,ParkName,Year,RecreationVisits,VisitorSpending,JobsSupported,LaborIncome,ValueAdded,EconomicOutput
0,Abraham Lincoln Birthplace NHP,2024,249166.0,17534000.0,195.0,7968000.0,14052000.0,24349000.0
1,Acadia NP,2024,3961661.0,538906000.0,5304.0,243133000.0,459921000.0,744545000.0
2,Adams NHP,2024,45656.0,3213000.0,31.0,1804000.0,3028000.0,4691000.0
3,African Burial Ground NM,2024,36007.0,2534000.0,21.0,1410000.0,2447000.0,3606000.0
4,Agate Fossil Beds NM,2024,19395.0,1694000.0,18.0,484000.0,960000.0,1813000.0


## Step 5: Getting Previous Visitor Spending Data (2017-2023)

In [44]:
year_pages = {
    2017: (24,37),          #indexing starts from 0 so start =page-1 and end =page+1 cause in range it's excluded whne iterating
    2018: (25,44),
    2019: (23,41),
    2020: (25,46),
    2021: (25,49),
    2022: (26,50),
    2023: (26,46)
}

In [45]:
#year_pages = {    2017: (24,37),    2018: (25,44),    2019: (23,41),    2020: (25,46),    2021: (25,49),    2022: (26,50),    2023: (26,46)}
pdf_path = os.path.join(raw_path, 'NPS_2020_Visitor_Spending_Effects.pdf')
with pdfplumber.open(pdf_path) as pdf:
    text = pdf.pages[25].extract_text()
    print(text[:500])

Appendix
Table A-1. NPS visits, spending, and economic contributions to local economies – 2020.
Total Visitor Economic
Total Recreation Spending Labor Income Value Added Output
Park Unit Visits ($000s, $2020) Jobs ($000s, $2020) ($000s, $2020) ($000s, $2020)
Abraham Lincoln Birthplace NHP 228,140 $13,669 202 $6,138 $10,709 $18,484
Acadia NPa 2,669,034 $303,734 4,368 $134,853 $240,528 $411,030
Adams NHP 6,937 $416 5 $231 $379 $595
African Burial Ground NM 7,907 $474 6 $262 $443 $661
Agate Fossil 


In [46]:
def process_vse_pdf(year, raw_path):
    pdf_path = os.path.join(raw_path, f'NPS_{year}_Visitor_Spending_Effects.pdf')
    start, end = year_pages[year]
    
    all_rows = []
    
    # try table extraction first (works for 2017,2018,2019,2023)
    with pdfplumber.open(pdf_path) as pdf:
        for i in range(start, end):
            table = pdf.pages[i].extract_table()
            if table:
                # skip header rows, keep data rows only
                for row in table:
                    if row and row[0] and row[0] != 'Park Unit':
                        all_rows.append(row)
    
    # fallback: regex parse text (works for 2020,2021,2022)
    if not all_rows:
        print(f'[{year}] table extraction failed, falling back to regex')
        # pattern: anything (park name) + 6 numeric columns
        # numeric = optional $, digits and commas, optional decimals
        pattern = re.compile(
            r'^(.+?)\s+([\d,]+)\s+\$?([\d,]+)\s+([\d,]+)\s+\$?([\d,]+)\s+\$?([\d,]+)\s+\$?([\d,]+)\s*$'
        )
        with pdfplumber.open(pdf_path) as pdf:
            for i in range(start, end):
                text = pdf.pages[i].extract_text() or ''
                for line in text.split('\n'):
                    match = pattern.match(line.strip())
                    if match:
                        all_rows.append(list(match.groups()))
    
    if not all_rows:
        print(f'[{year}] BOTH approaches failed')
        return None
    
    cols = ['ParkName','RecreationVisits','VisitorSpending',
            'JobsSupported','LaborIncome','ValueAdded','EconomicOutput']
    df = pd.DataFrame(all_rows, columns=cols)
    df.insert(1, 'Year', year)
    print(f'[{year}] extracted {len(df)} rows')
    return df

In [47]:
historical_dfs = []
for year in [2017, 2018, 2019, 2020, 2021, 2022, 2023]:
    result = process_vse_pdf(year, raw_path)
    if result is not None:
        historical_dfs.append(result)

vse_historical_df = pd.concat(historical_dfs, ignore_index=True)
print(f'\nShape of VSE historical data: {vse_historical_df.shape}')
print(f'\nRows per year:\n{vse_historical_df.groupby("Year").size()}')

[2017] extracted 382 rows
[2018] extracted 382 rows
[2019] extracted 382 rows
[2020] table extraction failed, falling back to regex
[2020] extracted 386 rows
[2021] table extraction failed, falling back to regex
[2021] extracted 391 rows
[2022] table extraction failed, falling back to regex
[2022] extracted 392 rows
[2023] extracted 397 rows

Shape of VSE historical data: (2712, 8)

Rows per year:
Year
2017    382
2018    382
2019    382
2020    386
2021    391
2022    392
2023    397
dtype: int64


In [48]:
# 1. Strip parentheticals (handles "(New Mexico)", "(Tennessee)", "(x)", etc.)
vse_historical_df['ParkName'] = vse_historical_df['ParkName'].str.replace(r'\s*\([^)]*\)\s*', ' ', regex=True)

# 2. trailing letter footnotes A-D and a-d, with or without space, with or without commas
#    matches: " A", " A,B", "A", "abc", "a,b,c", "NPA", "NPABC"
vse_historical_df['ParkName'] = vse_historical_df['ParkName'].str.replace(r'\s*[A-Ca-c](?:,?[A-Ca-c]){0,3}$', '', regex=True)

# 3. trailing symbol footnotes: *, !, with or without space
vse_historical_df['ParkName'] = vse_historical_df['ParkName'].str.replace(r'\s*[\*!]+\s*$', '', regex=True)

# 4. ampersand spacing fix
vse_historical_df['ParkName'] = vse_historical_df['ParkName'].str.replace(r'&(?!\s)', '& ', regex=True)
vse_historical_df['ParkName'] = vse_historical_df['ParkName'].str.replace(r'(?<!\s)&', ' &', regex=True)

# 5. final whitespace strip
vse_historical_df['ParkName'] = vse_historical_df['ParkName'].str.strip()

print('Sample after cleaning:')
vse_historical_df.sample(10)

Sample after cleaning:


,ParkName,Year,RecreationVisits,VisitorSpending,JobsSupported,LaborIncome,ValueAdded,EconomicOutput
384,Adams NHP,2018,"121,007","$7,146",94,"$3,954","$6,492","$10,196"
877,Fort Laramie NHS,2019,"42,893","$2,551",37,$912,"$1,642","$3,016"
635,National Capital Parks East,2018,"1,447,273","$27,687",371,"$15,107","$25,716","$40,603"
2310,Wupatki NM,2022,"194,448","14,153",170,"5,726","9,504","16,660"
455,Clara Barton NHS,2018,425,$25,0,$13,$22,$35
2646,Santa Monica Mountains NR,2023,"759,352","$38,448",444,"$21,420","$33,702","$54,794"
150,Governors Island NM,2017,"625,652","$35,770.7",422,"$20,182.7","$32,645.8","$48,914.2"
1228,Cumberland Gap NHP,2020,"735,447","47,992",653,"20,516","35,278","61,463"
165,Hampton NHS,2017,"32,328","$1,848.3",25,$976.6,"$1,624.6","$2,581.6"
1720,Jewel Cave NM,2021,"108,209","7,044",98,"2,870","4,975","9,287"


In [49]:
temp_short_names = vse_historical_df[vse_historical_df['ParkName'].str.len() < 5]['ParkName'].unique()
print(f'Suspiciously short names: {temp_short_names}')

# also check: any park name that exists in 2024 (already clean) but somehow a different-length version exists in 2017-2023
temp_parks_historical = set(vse_historical_df['ParkName'].unique())
matches = temp_irma_parks & temp_parks_historical
print(f'\nParks matching 2024 cleaned names: {len(matches)} out of {len(temp_parks_historical)} historical unique')

Suspiciously short names: <StringArray>
[]
Length: 0, dtype: str

Parks matching 2024 cleaned names: 327 out of 489 historical unique


In [50]:
money_cols = ['VisitorSpending','LaborIncome','ValueAdded','EconomicOutput']
count_cols = ['RecreationVisits','JobsSupported']

for col in money_cols:
    vse_historical_df[col] = (vse_historical_df[col].astype(str)
                              .str.replace(',', '', regex=False)
                              .str.replace('$', '', regex=False)
                              .str.replace('#', '', regex=False)
                              .astype(float) * 1000)

for col in count_cols:
    vse_historical_df[col] = (vse_historical_df[col].astype(str)
                              .str.replace(',', '', regex=False)
                              .str.replace('#', '', regex=False)
                              .astype(float))

print(vse_historical_df.dtypes)
print(f'\nNulls per col:\n{vse_historical_df.isnull().sum()}')

ParkName                str
Year                  int64
RecreationVisits    float64
VisitorSpending     float64
JobsSupported       float64
LaborIncome         float64
ValueAdded          float64
EconomicOutput      float64
dtype: object

Nulls per col:
ParkName            0
Year                0
RecreationVisits    0
VisitorSpending     0
JobsSupported       0
LaborIncome         0
ValueAdded          0
EconomicOutput      0
dtype: int64


In [51]:
# # find all values containing # across money columns
# for col in ['VisitorSpending','LaborIncome','ValueAdded','EconomicOutput']:
#     bad = vse_historical_df[vse_historical_df[col].astype(str).str.contains('#', na=False)]
#     if len(bad) > 0:
#         print(f'\n{col}: {len(bad)} rows')
#         print(bad[['ParkName','Year',col]].head(10))

#if we need to replace ANY unknown character
for col in money_cols + count_cols:
    sample = vse_historical_df[col].astype(str).str.cat(sep=' ')
    weird = set(re.findall(r'[^\d,.\$\#\s\-]', sample))
    print(f'{col}: weird chars = {weird}')

VisitorSpending: weird chars = set()
LaborIncome: weird chars = set()
ValueAdded: weird chars = set()
EconomicOutput: weird chars = set()
RecreationVisits: weird chars = set()
JobsSupported: weird chars = set()


In [52]:
#combining this dataset with 2024 VSE data
vse_master_df = pd.concat([vse_2024_df, vse_historical_df], ignore_index=True)
vse_master_df = vse_master_df.sort_values(['ParkName','Year']).reset_index(drop=True)
print(f'Master shape: {vse_master_df.shape}')
print(f'\nRows per year:\n{vse_master_df.groupby("Year").size()}')

Master shape: (3110, 8)

Rows per year:
Year
2017    382
2018    382
2019    382
2020    386
2021    391
2022    392
2023    397
2024    398
dtype: int64


In [53]:
irma_parks = set(visit_df['ParkName'].unique())
vse_parks = set(vse_master_df['ParkName'].unique())
unmatched = vse_parks - irma_parks

print(f'VSE total unique parks: {len(vse_parks)}')
print(f'Matched: {len(vse_parks & irma_parks)}')
print(f'Unmatched: {len(unmatched)}\n')

fuzzy_results = []
for vse_name in unmatched:
    match, score, _ = process.extractOne(vse_name, irma_parks, scorer=fuzz.ratio)
    fuzzy_results.append({'vse_name': vse_name, 'irma_match': match, 'score': score})

fuzzy_df = pd.DataFrame(fuzzy_results).sort_values('score')

print('\n=== Below 75 (manual decision) ===')
print(fuzzy_df[fuzzy_df['score'] < 75].to_string(index=False))
print('\n=== 75-89 (verify) ===')
print(fuzzy_df[(fuzzy_df['score'] >= 75) & (fuzzy_df['score'] < 90)].to_string(index=False))
print('=== 90+ (auto accept) ===')
print(fuzzy_df[fuzzy_df['score'] >= 90].to_string(index=False))

VSE total unique parks: 560
Matched: 398
Unmatched: 162


=== Below 75 (manual decision) ===
                                                 vse_name                                 irma_match     score
                      President William Jefferson Clinton                 Prince William Forest Park 52.459016
                  Lyndon Baines Johnson Memorial Grove on          LBJ Memorial Grove on the Potomac 58.333333
                     Lyndon Baines Johnson Memorial Grove                      Lyndon B. Johnson NHP 59.649123
                          Wolf Trap National Park for the       Wolf Trap NP for the Performing Arts 59.701493
                    World War II Valor in the Pacific\nNM                     War in the Pacific NHP 65.517241
                           Lyndon Baines Johnson Memorial                      Lyndon B. Johnson NHP 66.666667
                                              Ocmulgee NM                        Ocmulgee Mounds NHP 66.666667
                   

In [54]:
vse_master_df[vse_master_df[['ParkName','Year']].duplicated()]

,ParkName,Year,RecreationVisits,VisitorSpending,JobsSupported,LaborIncome,ValueAdded,EconomicOutput
1840,Manhattan Project NHP,2017,12172.0,391500.0,6.0,139900.0,254400.0,423200.0
1841,Manhattan Project NHP,2017,70406.0,1309600.0,19.0,529400.0,826500.0,1456800.0
1843,Manhattan Project NHP,2018,31640.0,607000.0,8.0,256000.0,402000.0,684000.0
1844,Manhattan Project NHP,2018,23742.0,788000.0,9.0,289000.0,531000.0,863000.0
1846,Manhattan Project NHP,2019,30123.0,581000.0,8.0,246000.0,386000.0,657000.0
1847,Manhattan Project NHP,2019,27958.0,935000.0,11.0,343000.0,631000.0,1025000.0
1849,Manhattan Project NHP,2020,264.0,9000.0,0.0,3000.0,6000.0,10000.0
1851,Manhattan Project NHP,2021,18479.0,392000.0,5.0,169000.0,263000.0,481000.0
1852,Manhattan Project NHP,2021,0.0,0.0,0.0,0.0,0.0,0.0
1854,Manhattan Project NHP,2022,10546.0,227000.0,3.0,98000.0,152000.0,278000.0


In [55]:
# manual overrides for fuzzy matches that were wrong
manual_overrides = {
    'Jefferson NEM': 'Gateway Arch NP',                                     # residgnated in 2018
    'Fort Sumter NM': 'Fort Sumter and Fort Moultrie NHP',
    'World War II Valor in the Pacific NM': 'Pearl Harbor NMEM',            # redesignated in 2019
    'World War II Valor in the Pacific NM *': 'Pearl Harbor NMEM',
    'World War II Valor in the Pacific\nNM': 'Pearl Harbor NMEM',
    'Longfellow NHS': "Longfellow House Washington's HQ NHS",
    'Lyndon Baines Johnson Memorial': 'LBJ Memorial Grove on the Potomac',
    'Lyndon Baines Johnson Memorial Grove': 'LBJ Memorial Grove on the Potomac',
    'Lyndon Baines Johnson Memorial Grove on': 'LBJ Memorial Grove on the Potomac',
    'President William Jefferson Clinton': 'President W.J. Clinton Birthplace Home NHS',
    'Manhattan Project': 'Manhattan Project NHP',                                               #was wrapped into 2 columns in 2020 pdf so NHP didn't get read, will consolidate down
}


# build dict from fuzzy results, drop the rows you manually overrode
auto_mapping = dict(zip(fuzzy_df['vse_name'], fuzzy_df['irma_match']))

# combine
historical_name_mapping = {**auto_mapping, **manual_overrides}         #The ** is dict unpacking. It merges two dicts. If a key exists in both, the second one wins. So this combines the auto and manual mappings into one final dict.

vse_master_df['ParkName'] = vse_master_df['ParkName'].replace(historical_name_mapping)

# Verifying all parks exist
still_unmatched = set(vse_master_df['ParkName']) - irma_parks
print(f'Still unmatched: {len(still_unmatched)}')
for p in sorted(still_unmatched):
    print(f'  {p}')

# Consolidate duplicates ONCE
print(f'Shape before grouping duplicates: {vse_master_df.shape}')
vse_master_df = vse_master_df.groupby(['ParkName','Year'], as_index=False).sum()
print(f'Final shape: {vse_master_df.shape}')

Still unmatched: 0
Shape before grouping duplicates: (3110, 8)
Final shape: (3082, 8)


In [56]:
vse_master_df[vse_master_df['ParkName']=='Manhattan Project NHP']

,ParkName,Year,RecreationVisits,VisitorSpending,JobsSupported,LaborIncome,ValueAdded,EconomicOutput
1831,Manhattan Project NHP,2017,89598.0,2102500.0,29.0,833800.0,1352000.0,2343200.0
1832,Manhattan Project NHP,2018,69543.0,2231000.0,28.0,888000.0,1505000.0,2525000.0
1833,Manhattan Project NHP,2019,79871.0,2812000.0,36.0,1121000.0,1905000.0,3200000.0
1834,Manhattan Project NHP,2020,11391.0,287000.0,3.0,119000.0,192000.0,328000.0
1835,Manhattan Project NHP,2021,26063.0,886000.0,11.0,376000.0,605000.0,1078000.0
1836,Manhattan Project NHP,2022,27265.0,1126000.0,14.0,467000.0,775000.0,1348000.0
1837,Manhattan Project NHP,2023,60100.0,2492000.0,31.0,1031000.0,1714000.0,2984000.0
1838,Manhattan Project NHP,2024,74693.0,5335000.0,49.0,2276000.0,4152000.0,6713000.0


In [57]:
vse_master_df.to_csv(os.path.join(processed_path, 'VSE_2017_2024_processed.csv'), index=False)